In [14]:
import cv2
import numpy as np
import os

from google.colab import files
from IPython.display import Video, display

In [15]:
os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [23]:
uploaded = files.upload()

Saving cvlab.mp4 to cvlab.mp4


In [24]:
import shutil

uploaded_filename = list(uploaded.keys())[0]

shutil.move(
    uploaded_filename,
    "/content/data/cvlab.mp4"
)

print("Video saved as: data/cvlab.mp4")

Video saved as: data/cvlab.mp4


In [25]:
#color balance

def color_balance(image):

    # Convert image to floating point for calculations
    img = image.astype(np.float32)

    # Separate the Blue, Green and Red channels
    b, g, r = cv2.split(img)

    # Calculate average intensity of each channel
    avg_b = np.mean(b)
    avg_g = np.mean(g)
    avg_r = np.mean(r)

    # Overall average intensity
    avg_gray = (avg_b + avg_g + avg_r) / 3

    # Scale each channel toward the common average
    b = b * (avg_gray / (avg_b + 1e-6))
    g = g * (avg_gray / (avg_g + 1e-6))
    r = r * (avg_gray / (avg_r + 1e-6))

    # Merge channels again
    balanced = cv2.merge([b, g, r])

    # Make sure pixel values stay between 0 and 255
    balanced = np.clip(balanced, 0, 255)

    return balanced.astype(np.uint8)

In [26]:
#log transform

def log_transform(image):

    # Convert to floating point
    image_float = image.astype(np.float32)

    # Scaling constant
    c = 255 / np.log(1 + 255)

    # Apply logarithmic transformation
    log_image = c * np.log(1 + image_float)

    # Restrict values to valid image range
    log_image = np.clip(log_image, 0, 255)

    return log_image.astype(np.uint8)

In [27]:
def power_law_transform(image, gamma=1.4):

    # Normalize image from 0-255 to 0-1
    normalized = image.astype(np.float32) / 255.0

    # Apply power-law transformation
    transformed = np.power(normalized, gamma)

    # Convert back to 0-255
    transformed = transformed * 255

    transformed = np.clip(transformed, 0, 255)

    return transformed.astype(np.uint8)

In [28]:
video_path = "data/cvlab.mp4"

# Initialize VideoCapture
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open the video.")

else:
    print("Video opened successfully.")


# Get video information
fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


print("FPS:", fps)
print("Width:", width)
print("Height:", height)
print("Total Frames:", frame_count)

Video opened successfully.
FPS: 50.0
Width: 112
Height: 112
Total Frames: 140


Frame

 ↓

Histogram Equalization

 ↓

JET Color Mapping

 ↓

Color Balance

 ↓

Logarithmic Transformation

 ↓

Power-Law Transformation

 ↓

Side-by-Side Monitoring Array

In [29]:
# Temporary output video
temp_output = "output/processed_echo_temp.mp4"

# Since raw + enhanced are placed horizontally,
# the final width is twice the original width
output_width = width * 2

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    temp_output,
    fourcc,
    fps,
    (output_width, height)
)


processed_frames = 0


while True:

    # ---------------------------------------------
    # Read one frame from the ultrasound video
    # ---------------------------------------------
    ret, frame = cap.read()

    if not ret:
        break


    # Keep original frame for comparison
    raw_frame = frame.copy()


    # ---------------------------------------------
    # 1. Convert frame to grayscale
    # ---------------------------------------------
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )


    # ---------------------------------------------
    # 2. Histogram Equalization
    # ---------------------------------------------
    equalized = cv2.equalizeHist(gray)


    # ---------------------------------------------
    # 3. Convert to JET colored heatmap
    # ---------------------------------------------
    heatmap = cv2.applyColorMap(
        equalized,
        cv2.COLORMAP_JET
    )


    # ---------------------------------------------
    # 4. Apply Color Balance
    # ---------------------------------------------
    balanced = color_balance(heatmap)


    # ---------------------------------------------
    # 5. Apply Logarithmic Transformation
    # ---------------------------------------------
    log_image = log_transform(balanced)


    # ---------------------------------------------
    # 6. Apply Power-Law Transformation
    # ---------------------------------------------
    enhanced = power_law_transform(
        log_image,
        gamma=1.4
    )


    # ---------------------------------------------
    # Add labels
    # ---------------------------------------------
    cv2.putText(
        raw_frame,
        "Raw Ultrasound",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    cv2.putText(
        enhanced,
        "Enhanced Ultrasound",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    # ---------------------------------------------
    # 7. Monitoring Array
    #
    # Raw frame | Enhanced frame
    # ---------------------------------------------
    comparison = np.hstack(
        (raw_frame, enhanced)
    )


    # Save processed frame
    writer.write(comparison)


    processed_frames += 1


# Release video resources
cap.release()
writer.release()


print("Processing completed.")
print("Frames processed:", processed_frames)

Processing completed.
Frames processed: 140


In [31]:
!ffmpeg -y \
-i output/processed_echo_temp.mp4 \
-vcodec libx264 \
output/processed_echo.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [32]:
display(
    Video(
        "output/processed_echo.mp4",
        embed=True,
        width=1000
    )
)